# Comparing neuron activations between two trained models

Two networks trained to similar loss can still compute their answer in completely
different internal ways. This notebook asks whether they do — for example, does an
evolutionary optimizer arrive at the same internal representation as gradient descent?

**Why this needs more than "diff the activations".** Hidden units have no canonical
identity:

- **Permutation** — unit 7 in one model may play the role of unit 412 in another.
- **Scale** — with ReLU you can multiply a unit's incoming weights by `a > 0` and its
  outgoing weights by `1/a` and get an identical function, so raw magnitudes aren't
  comparable.

So comparing unit *i* of A against unit *i* of B is meaningless. Instead:

| Tool | What it answers | Handles |
|---|---|---|
| **Linear CKA** | Are these two layers the same representation overall? | rotation + scaling (so also permutation) |
| **Neuron matching** | Which unit in B corresponds to unit *i* in A? | permutation, explicitly |
| **Dead-unit fraction** | How much capacity is wasted (ReLU stuck off)? | — |
| **Effective rank** | How many dimensions does the layer actually use? | — |

CKA near 1.0 means the two layers encode the same information up to rotation. A high
mean matched-correlation means the same *individual features* were learned, just
reordered. CKA high **but** matched-correlation low is the interesting case: the same
information, distributed differently across units.

In [ ]:
# Run from the repo root so repo-relative paths ("outputs/...", "data/...") resolve the
# same way they do for the CLI tools, regardless of where Jupyter was started.
import os
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(root)
print("working directory:", Path.cwd())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from optim_env.analysis.loading import load_run, load_dataset, predict
from optim_env.analysis.activations import hidden_activations
from optim_env.analysis.similarity import (
    compare_layers, linear_cka, match_neurons,
    dead_unit_fraction, effective_rank, _abs_correlation,
)

plt.rcParams["figure.dpi"] = 110

## 1. Pick two runs

Any two run directories under `outputs/runs/`. The layout is
`outputs/runs/<dataset>/<framework>_<method>/<algorithm>/<variant>/seed_<n>/`, so the
interesting comparisons are easy to express:

- **optimizer family** — `jax_SGD/adamw/baseline` vs `jax_NE/PSO/baseline`
- **HPO effect** — `jax_NE/PSO/baseline` vs `jax_NE/PSO/hpo`
- **seed variability** (the control) — `.../baseline/seed_0` vs `.../baseline/seed_1`

That last one matters: it's the baseline for how much *any* two runs differ. Compare
your cross-optimizer number against it before calling a difference meaningful.

In [ ]:
RUN_A = "outputs/runs/california/jax_NE/SNES/baseline/seed_0"
RUN_B = "outputs/runs/california/jax_NE/SNES/hpo/seed_0"

run_a, run_b = load_run(RUN_A), load_run(RUN_B)
print("A:", run_a.label)
print("B:", run_b.label)

## 2. Feed both models the same inputs

Activations only mean something relative to a fixed set of inputs, so both models see
the identical batch — the held-out test split, which neither one trained on.

In [ ]:
dataset = load_dataset(run_a.cfg)
X = dataset.get_test_ds().X[:1024]

acts_a = hidden_activations(run_a, X)
acts_b = hidden_activations(run_b, X)

for name, act in acts_a.items():
    print(f"{name}: {act.shape}  (samples x units)")

## 3. Layer-by-layer comparison

`compare_layers` runs every metric at once. Read it as: does the representation match
(CKA), do individual units match (matched_corr), and is either model wasting capacity
(dead_frac, eff_rank vs unit count)?

In [ ]:
import pandas as pd

summary = pd.DataFrame(compare_layers(acts_a, acts_b))
summary.round(4)

## 4. Are the same features there, just reordered?

Neuron matching undoes the permutation: each unit of A is assigned its best partner in
B, maximizing total correlation. The distribution of those correlations is the story.

- mass near **1.0** → same features, different order
- mass near **0** → genuinely different features
- a **split** distribution → a shared core plus model-specific units

In [ ]:
fig, axes = plt.subplots(1, len(acts_a), figsize=(4 * len(acts_a), 3.2), squeeze=False)

for ax, (name, A) in zip(axes[0], acts_a.items()):
    _, corrs = match_neurons(A, acts_b[name])
    ax.hist(corrs, bins=30, range=(0, 1), color="steelblue", edgecolor="white")
    ax.axvline(np.median(corrs), color="crimson", ls="--",
               label=f"median {np.median(corrs):.2f}")
    ax.set_title(name)
    ax.set_xlabel("|correlation| of matched units")
    ax.legend()

axes[0][0].set_ylabel("number of units")
fig.suptitle(f"Matched-unit correlation: {run_a.label}  vs  {run_b.label}", y=1.04)
plt.tight_layout()

## 5. Where does each unit's partner live?

The correlation matrix before matching. If both models learned the same features in the
same order you'd see a diagonal; in practice you get a scattered pattern, which is
exactly the permutation problem made visible.

In [ ]:
LAYER = list(acts_a)[0]
corr = _abs_correlation(acts_a[LAYER], acts_b[LAYER])

# Reorder B's units by the optimal matching -- a clean diagonal after reordering means
# the matching found a genuine one-to-one correspondence.
pairs, _ = match_neurons(acts_a[LAYER], acts_b[LAYER])
order = pairs[:, 1]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.6))
for ax, mat, title in [(ax1, corr, "as trained"), (ax2, corr[:, order], "B reordered by matching")]:
    im = ax.imshow(mat[:120, :120], cmap="magma", vmin=0, vmax=1, aspect="auto")
    ax.set_title(f"{LAYER}, {title}")
    ax.set_xlabel("unit in B"); ax.set_ylabel("unit in A")
    fig.colorbar(im, ax=ax, label="|correlation|")
plt.tight_layout()

## 6. Wasted capacity

Two failure modes worth checking, especially for evolutionary optimizers, which get no
gradient signal to revive a unit once it stops firing:

- **dead units** — never activate on any input, so they contribute nothing
- **effective rank** — how many dimensions the layer really spans; far below the unit
  count means the layer collapsed onto a small subspace no matter how wide it is

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.6))
names = list(acts_a)
x = np.arange(len(names))

ax1.bar(x - 0.2, [dead_unit_fraction(acts_a[n]) for n in names], 0.4, label=run_a.label)
ax1.bar(x + 0.2, [dead_unit_fraction(acts_b[n]) for n in names], 0.4, label=run_b.label)
ax1.set_xticks(x); ax1.set_xticklabels(names); ax1.set_ylabel("fraction of dead units")
ax1.set_title("Dead ReLU units"); ax1.legend(fontsize=8)

width = [acts_a[n].shape[1] for n in names]
ax2.bar(x - 0.2, [effective_rank(acts_a[n]) for n in names], 0.4, label=run_a.label)
ax2.bar(x + 0.2, [effective_rank(acts_b[n]) for n in names], 0.4, label=run_b.label)
ax2.plot(x, width, "k_", markersize=22, label="units available")
ax2.set_xticks(x); ax2.set_xticklabels(names); ax2.set_ylabel("effective rank")
ax2.set_title("Dimensions actually used"); ax2.legend(fontsize=8)
plt.tight_layout()

## 7. Calibrate against the seed baseline

A CKA of 0.9 means nothing on its own. Compare it against two same-optimizer runs that
differ **only** by seed — that's the floor for "different by accident". Anything at or
above it is not a real representational difference.

In [ ]:
COMPARISONS = {
    # label: (run_a, run_b)
    "same optimizer, different seed": (
        "outputs/runs/california/jax_NE/SNES/baseline/seed_0",
        "outputs/runs/california/jax_NE/SNES/baseline/seed_1",
    ),
    "baseline vs HPO-tuned": (RUN_A, RUN_B),
}

rows = []
for label, (pa, pb) in COMPARISONS.items():
    try:
        ra, rb = load_run(pa), load_run(pb)
        aa, ab = hidden_activations(ra, X), hidden_activations(rb, X)
        for name in aa:
            rows.append({"comparison": label, "layer": name, "cka": linear_cka(aa[name], ab[name])})
    except FileNotFoundError as e:
        print(f"skipping '{label}': {e}")

pd.DataFrame(rows).pivot(index="layer", columns="comparison", values="cka").round(4)

### Reading the result

If cross-optimizer CKA sits meaningfully **below** the seed-to-seed number, the two
optimizers really did find different internal solutions — which is the substantive claim
this repository is set up to test. If it sits at the same level, they converged to the
same representation by different routes, and the difference is only in the path taken.